# AcusTera · optional remote GPU processing (DiariZen + CrisperWhisper)

This notebook runs the two heavy speech engines of AcusTera on a Google Colab GPU and produces, for each audio file,
`<name>_diarizen.json` / `.rttm` (who speaks when) and `<name>_crisperwhisper.json` (verbatim words with timestamps).
In AcusTera, point **03 · Process → Precomputed results** to the folder with these files; role assignment, acoustic
analysis and statistics then run on your own computer.

> ## ⚠️ Privacy: with this notebook your data are no longer only on your computer
> AcusTera is designed to work **entirely locally**. This notebook is an **optional** shortcut for computers without an
> NVIDIA GPU, where DiariZen and CrisperWhisper are slow (on a Mac they run on the Apple GPU or the CPU, far slower
> than on a cloud GPU). Using it means that:
> - the **audio recordings are uploaded to Google's servers** (the Colab virtual machine and, if you enable it, Google Drive);
> - the **results contain the transcription of what was said**, which is personal data, and possibly special-category
>   (health) data in clinical recordings;
> - processing happens under Google's terms of service, possibly outside your country or the EU.
>
> **Before using it**, check that the participants' consent, your ethics approval and your institution's data
> protection rules allow processing by this external provider. If in doubt, **do not use it**: run the engines locally
> from AcusTera, or use pyannote 3.1 + Whisper, which are lighter.
>
> If you use it: give the files neutral names (e.g. `S01.mp3`, no names or dates), convert them to 16 kHz mono MP3 to
> upload only what is needed, keep `SAVE_TO_DRIVE = False`, and run the last cell to delete everything from the virtual
> machine when you finish.

## 1 · Check the GPU
*Runtime → Change runtime type → GPU*. Without a GPU the engines are very slow.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 · Settings

In [ ]:
import os
AUDIO   = '/content/audio'      # drag your audio files here (Files panel, left)
RESULTS = '/content/results'    # output folder on the virtual machine
N_SPEAKERS = 3                  # couples therapy: man, woman, therapist
LANGUAGE   = 'es'               # language code, or 'auto'
SAVE_TO_DRIVE = False           # True copies the results to your Google Drive (they then stay there)
os.makedirs(AUDIO, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUT = '/content/drive/MyDrive/acustera_results'
    os.makedirs(DRIVE_OUT, exist_ok=True)
os.environ['MPLBACKEND'] = 'Agg'
print('Upload the audio files to', AUDIO, 'and wait until the upload finishes.')

## 3 · Install the engines (≈10 min)
Same versions as AcusTera: CrisperWhisper 2.0.3; DiariZen at commit 844f5555 with PyTorch 2.8.

In [ ]:
%%bash
set -e
pip -q install uv
uv venv -q /content/cw --python 3.12
uv pip install --python /content/cw/bin/python -q "crisperwhisper[ct2,convert]==2.0.3" "transformers>=4.46,<5"
[ -d /content/DiariZen ] || git clone -q --recurse-submodules https://github.com/BUTSpeechFIT/DiariZen /content/DiariZen
cd /content/DiariZen && git checkout -q 844f5555b0a98acd0931511fc641a8c5b8ba92c7 && git submodule update -q --init --recursive
uv venv -q /content/dz --python 3.10
grep -vE 'onnxruntime|jupyterlab|pre-commit' requirements.txt > /tmp/req_dz.txt
uv pip install --python /content/dz/bin/python -q torch==2.8.0 torchaudio==2.8.0 numpy==1.26.4 -r /tmp/req_dz.txt -r pyannote-audio/requirements.txt
uv pip install --python /content/dz/bin/python -q -e pyannote-audio -e .
/content/cw/bin/python -c "import crisperwhisper, ctranslate2; print('CrisperWhisper', crisperwhisper.__version__, 'CUDA:', ctranslate2.get_cuda_device_count())"
/content/dz/bin/python -c "import torch, diarizen; print('DiariZen OK, torch', torch.__version__, 'CUDA:', torch.cuda.is_available())"

Model revisions used to validate AcusTera (a warning means Hugging Face now serves a newer version):

In [ ]:
from huggingface_hub import HfApi
for repo, sha in {'nyralabs/CrisperWhisper2.0_large': 'f4334f6e8193f2691212d49b20fa12d370e13896',
                  'BUT-FIT/diarizen-wavlm-large-s80-md-v2': '027b3e221b9d81dce2be794084f1dbd3ba2da403'}.items():
    cur = HfApi().model_info(repo).sha
    print(('OK      ' if cur == sha else 'WARNING newer revision ') + repo, cur)

## 4 · Engine scripts (identical to the ones AcusTera runs locally)

In [ ]:
%%writefile /content/engine_diarizen.py
# AcusTera engine script: DiariZen diarization (same format as the Colab notebook)
# usage: python engine_diarizen.py AUDIO START_S END_S OUT.json N_SPEAKERS
import json, os, subprocess, sys, tempfile, time
import torch
from diarizen.pipelines.inference import DiariZenPipeline
MODEL = 'BUT-FIT/diarizen-wavlm-large-s80-md-v2'

def main(audio, ini, fin, out, n='3'):
    ini, fin, n = float(ini), float(fin), int(n)
    with tempfile.TemporaryDirectory() as tmp:
        wav = os.path.join(tmp, 'chunk.wav')
        subprocess.run(['ffmpeg', '-nostdin', '-loglevel', 'error', '-ss', str(ini), '-to', str(fin),
                        '-i', audio, '-ac', '1', '-ar', '16000', wav], check=True)
        p = DiariZenPipeline.from_pretrained(MODEL)
        p.min_speakers = p.max_speakers = n
        modern = tuple(int(x) for x in torch.__version__.split('.')[:2]) >= (2, 4)
        dev = os.environ.get('DIARIZAR_DISPOSITIVO') or (
            'cuda' if torch.cuda.is_available() else
            'mps' if torch.backends.mps.is_available() and modern else 'cpu')
        p.to(torch.device(dev))
        t1 = time.time(); r = p(wav); dur = time.time() - t1
    turns = [{'hablante': f'SPK_{spk}', 'ini': round(t.start + ini, 3), 'fin': round(t.end + ini, 3)}
             for t, _, spk in r.itertracks(yield_label=True)]
    json.dump({'audio': os.path.basename(audio), 'modelo': MODEL, 'n_hablantes': n, 'dispositivo': dev,
               'inicio_s': ini, 'fin_s': fin, 'segundos_proceso': round(dur), 'turnos': turns},
              open(out, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    print(f'{len(turns)} turns; {fin - ini:.0f} s of audio in {dur:.0f} s ({dev})')

if __name__ == '__main__':
    main(*sys.argv[1:])

In [ ]:
%%writefile /content/engine_crisper.py
# AcusTera engine script: CrisperWhisper 2.0 verbatim transcription (same format as the Colab notebook)
# usage: python engine_crisper.py AUDIO START_S END_S OUT.json LANGUAGE
import json, os, subprocess, sys, time
import numpy as np
MODEL = 'nyralabs/CrisperWhisper2.0_large'
RETRIES = [{}, {'max_new_tokens': 192, 'context_words': 6}, {'max_new_tokens': 128, 'context_words': 0}]

def read_chunk(audio, ini, fin, sr=16000):
    raw = subprocess.run(['ffmpeg', '-nostdin', '-loglevel', 'error', '-ss', str(ini), '-to', str(fin), '-i', audio,
                          '-ac', '1', '-ar', str(sr), '-f', 'f32le', '-'], capture_output=True, check=True).stdout
    return np.frombuffer(raw, dtype=np.float32)

def main(audio, ini, fin, out, language='es'):
    import torch
    from crisperwhisper import CrisperWhisperModel
    ini, fin = float(ini), float(fin)
    if torch.cuda.is_available():
        dev = 'cuda'; m = CrisperWhisperModel(MODEL, backend='ct2', device='cuda', compute_type='float16')
    else:
        dev = 'mps' if torch.backends.mps.is_available() else 'cpu'
        m = CrisperWhisperModel(MODEL, backend='transformers', device=dev)
    wav = read_chunk(audio, ini, fin)
    kw = dict(sr=16000, mode='verbatim', word_timestamps=True)
    if language and language != 'auto': kw['language'] = language
    t1 = time.time()
    for k, extra in enumerate(RETRIES):
        try:
            r = m.transcribe(wav, **kw, **extra); break
        except RuntimeError as e:
            if 'position' not in str(e) or k == len(RETRIES) - 1: raise
    dur = time.time() - t1
    words = [{'palabra': w.word, 'ini': round(w.start + ini, 3), 'fin': round(w.end + ini, 3)} for w in r.words]
    json.dump({'audio': os.path.basename(audio), 'modelo': MODEL, 'modo': 'verbatim', 'inicio_s': ini,
               'fin_s': fin, 'segundos_proceso': round(dur), 'parametros': extra, 'texto': r.text,
               'palabras': words}, open(out, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    print(f'{len(words)} words; {fin - ini:.0f} s of audio in {dur:.0f} s ({dev})')

if __name__ == '__main__':
    main(*sys.argv[1:])

## 5 · Process all the audio files
Files already processed are skipped, so the cell can be re-run after an interruption.

In [ ]:
import subprocess, shutil, time
EXT = ('.mp3', '.wav', '.flac', '.m4a')
def duration(f):
    return float(subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', f],
                                capture_output=True, text=True, check=True).stdout)
def run(cmd, env=None):
    p = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, **(env or {})})
    print('   ', (p.stdout.strip().splitlines() or [''])[-1])
    if p.returncode: print(p.stderr[-2000:]); raise RuntimeError(cmd[1])
files = sorted(f for f in os.listdir(AUDIO) if f.lower().endswith(EXT))
print(len(files), 'audio files')
for n, f in enumerate(files, 1):
    stem, src = os.path.splitext(f)[0], os.path.join(AUDIO, f)
    out_t, out_d = f'{RESULTS}/{stem}_crisperwhisper.json', f'{RESULTS}/{stem}_diarizen.json'
    if os.path.exists(out_t) and os.path.exists(out_d): print(f'[{n}/{len(files)}] {stem}: done'); continue
    d = duration(src); print(f'[{n}/{len(files)}] {stem}: {d / 60:.1f} min')
    try:
        if not os.path.exists(out_t):
            run(['/content/cw/bin/python', '/content/engine_crisper.py', src, '0', f'{d:.3f}', out_t, LANGUAGE])
        if not os.path.exists(out_d):
            run(['/content/dz/bin/python', '/content/engine_diarizen.py', src, '0', f'{d:.3f}', out_d, str(N_SPEAKERS)],
                {'TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD': '1'})
            import json
            turns = json.load(open(out_d))['turnos']
            with open(out_d[:-5] + '.rttm', 'w') as fh:
                for t in turns:
                    fh.write(f"SPEAKER {stem} 1 {t['ini']:.3f} {t['fin'] - t['ini']:.3f} <NA> <NA> {t['hablante']} <NA> <NA>\n")
    except RuntimeError as e:
        print('    ERROR, continuing with the next file:', e)
    if SAVE_TO_DRIVE:
        for x in os.listdir(RESULTS):
            if x.startswith(stem + '_'): shutil.copy(f'{RESULTS}/{x}', DRIVE_OUT)
print('Finished')

## 6 · Download the results
Unzip them on your computer and point AcusTera's *Precomputed results* folder to them.

In [ ]:
import zipfile
from google.colab import files
with zipfile.ZipFile('/content/acustera_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for x in os.listdir(RESULTS): z.write(os.path.join(RESULTS, x), x)
files.download('/content/acustera_results.zip')

## 7 · Delete everything from the virtual machine
Run this when the download has finished. Then *Runtime → Disconnect and delete runtime*. If you enabled `SAVE_TO_DRIVE`, the copies in your Drive are **not** deleted by this cell.

In [ ]:
import shutil
for p in (AUDIO, RESULTS, '/content/acustera_results.zip'):
    if os.path.isdir(p): shutil.rmtree(p)
    elif os.path.exists(p): os.remove(p)
print('Audio and results deleted from the virtual machine.')